# Подготовка датасета

Цель: собрать/синтезировать датасет минимум из 200 instruction-response
примеров для дообучения модели под определённый стиль ответов — в нашем
случае "лаконичный эксперт": сохранение сути и фактов, устранение
многословности и вводных оговорок.

## Шаг 1: Загрузка и первичное знакомство с датасетом

Используем databricks-dolly-15k — датасет из ~15,000 instruction-response пар,
написанных сотрудниками Databricks (а не сгенерированных LLM). Этот датасет
примечателен тем, что тысячи разных сотрудников писали пары независимо друг
от друга, без единого редактора — из-за этого в данных могут встречаться
реальные человеческие ошибки и несогласованности (в отличие от более "стерильных"
датасетов вроде no_robots, курируемых небольшой командой профессиональных
аннотаторов). Начинаем с базового обзора: структура записей, распределение
по категориям.

In [ ]:
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split = "train")

# quick look at structure and volume before diving deeper
print(dataset[0])
print(len(dataset))

# sampling a few random indices to eyeball data quality/diversity
print(dataset[5])
print(dataset[200])

df = dataset.to_pandas()

# category distribution — determines which categories are usable
# for our "concise expert" style transfer goal
print(df["category"].value_counts())

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}
15011
{'instruction': 'If I have more pieces at the time of stalemate, have I won?', 'context': 'Stalemate is a situation in chess where the player whose turn it is to move is not in check and has no legal move. Stalemate results in a draw. During the endgame, stale

## Дедупликация

Раз данные писали независимо тысячи сотрудников, вероятны повторы. Проверяем
на двух уровнях: точный дубликат (одинаковый вопрос И ответ) и дубликат
только по вопросу (может указывать на повторяющийся шаблонный вопрос, но
с разными ответами — требует отдельного разбора, не всегда мусор).

In [ ]:
# strict duplicate: same instruction AND same response = definite junk

exact_dupes = df.duplicated(subset = ["instruction", "response"]).sum()
print(f"Exact Duplicates (instruction+response): {exact_dupes}")

# same instruction only - NOT necessarily junk, since response may differ
# (e.g. different context attached) - flagged for further investigation, not auto-removed
instr_dupes = df.duplicated(subset=["instruction"]).sum()
print(f"Duplicates according to instruction (may be various response): {instr_dupes}")

# only removing confirmed exact duplicates at this stage
df_clean = df.drop_duplicates(subset=["instruction", "response"]).copy()

Exact Duplicates (instruction+response): 15
Duplicates according to instruction (may be various response): 232


In [ ]:
# check what fraction of each category has non-empty context —
# needed because duplicate instructions with different context
# are NOT true duplicates (see instr_dupes above)
print(df_clean.groupby('category')["context"].apply(lambda x: (x.str.len() > 0).mean()))

category
brainstorming             0.0
classification            0.0
closed_qa                 1.0
creative_writing          0.0
general_qa                0.0
information_extraction    1.0
open_qa                   0.0
summarization             1.0
Name: context, dtype: float64


In [ ]:
# keep=False marks ALL duplicate rows (not just the 2nd, 3rd, etc.) —
# needed here since we want to VIEW both/all copies side by side,
# not just count them

dup_instructions = df_clean[df_clean.duplicated(subset = ['instruction'], keep=False)]
sample = dup_instructions.sort_values('instruction').head(10)

for _, row in sample.iterrows():
  print("Instruction: ", row['instruction'][:80])
  print("Context: ", row['context'][:80] if row['context'] else "(Empty)")
  print("Category: ", row['category'])
  print("-----")

Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Are you going for a walk today?
Context:  (Empty)
Category:  general_qa
-----
Instruction:  Based on the information below, when might people have started baking bread?
Context:  Agriculture encompasses crop and livestock production, aquaculture, fisheries an
Category:  information_extraction
-----
Instruction:  Based on the information below, when might people have started baking bread?
Context:  Agriculture encompasses crop and livestock production, aquaculture, fisheries an
Category:  closed_qa
-----
Instruction:  Can

## Сужение до целевых категорий и точная дедупликация

Ограничиваемся тремя категориями, где style transfer в "лаконичный эксперт"
осмыслен: open_qa, closed_qa, summarization (creative_writing, brainstorming
и т.д. осознанно исключены — там лаконичность противоречит сути задания).

Также уточняем дедупликацию: раньше проверяли дубли по instruction, но
context тоже часть входа модели — два одинаковых вопроса с разным context
не являются настоящими дубликатами.

In [ ]:
target_categories = ['open_qa', 'closed_qa', 'summarization']
df_target = df_clean[df_clean['category'].isin(target_categories)].copy()

print(df_target['category'].value_counts())

def dedup_key(row):
    # combine instruction + context into one comparable string —
    # this way, identical instructions with DIFFERENT context are
    # correctly treated as distinct (non-duplicate) examples
  if row['context'] and row['context'].strip():
    return row['instruction'].strip().lower() + "| | |" + row['context'].strip().lower()
  return row['instruction'].strip().lower()

df_target['dedup_key'] = df_target.apply(dedup_key, axis=1)

# "|||" separator is arbitrary but must be unlikely to appear naturally in the text
dupes_in_target = df_target.duplicated(subset=['dedup_key']).sum()
print(f"Real duplicates (instruction + context) in our 3 categories: {dupes_in_target}")

df_final = df_target.drop_duplicates(subset = ['dedup_key']).copy()
print(f"Remains after dedup: {len(df_final)}")

category
open_qa          3737
closed_qa        1768
summarization    1187
Name: count, dtype: int64
Real duplicates (instruction + context) in our 3 categories: 42
Remains after dedup: 6650


## Диагностика: распределение длины ответов

Проверяем распределение response_len по категориям — ищем аномалии
(например, крайне длинные "summarization"-ответы могут сигнализировать
о структурных проблемах, а не просто о разнообразии текста). Также ищем
случаи, где response дословно совпадает с context — явный признак
ошибки при заполнении датасета (кто-то скопировал не то поле).

Это диагностический шаг, а не финальный фильтр — конкретное решение
о том, что делать с найденным, принимается ниже, по результатам осмотра.

In [ ]:
df_final['response_len'] = df_final['response'].str.split().str.len()
df_final['instruction_len'] = df_final['instruction'].str.split().str.len()

print(df_final.groupby('category')['response_len'].describe())

# manually inspect the longest responses per category —
# extreme outliers may indicate a data entry bug rather than genuine long-form content
for cat in ['closed_qa', 'open_qa', 'summarization']:
  sub = df_final[df_final['category'] == cat]
  top = sub.nlargest(5, 'response_len')

  # flag cases where response is literally identical to context —
  # likely a copy-paste error in the original dataset, not real summarization

  matches_context = sub.apply(
      lambda r: bool(r['context']) and r['response'].strip() == r['context'].strip(), axis=1
  )
  print(f"{cat}: {matches_context.sum()} from {len(sub)} responses match context")

  for _, row in top.iterrows():
    print(f"  response_len={row['response_len']} : {row['response'][:150]}")

  print()


                count       mean         std  min   25%   50%   75%     max
category                                                                   
closed_qa      1767.0  30.713073   48.232177  1.0  10.0  17.0  33.0   629.0
open_qa        3697.0  49.519881   71.271550  1.0   9.0  26.0  65.0  1230.0
summarization  1186.0  79.251265  154.496283  2.0  31.0  54.0  88.0  4274.0
closed_qa: 6 from 1767 responses match context
  response_len=629 : Water fluoridation is the controlled adjustment of fluoride to a public water supply solely to reduce tooth decay. Fluoridated water contains fluoride
  response_len=619 : General Data Protection Regulation provides guidelines on storing and processing personal data. Personal data is any information about an identified o
  response_len=602 : Observer bias is one of the types of detection bias and is defined as any kind of systematic divergence from accurate facts during observation and the
  response_len=471 : Roger Federer, born 8 August 1981, i

## Проверка summarization: compression ratio

Для summarization ожидаем, что response значительно короче context (иначе
это не сжатие, а пересказ той же длины). Считаем compression_ratio =
response_len / context_len и ищем аномально высокие значения (> 0.5).

Первая гипотеза: проблема связана с длиной context (слишком короткий
context делает ratio ненадёжным математически). Проверяем через ручной
осмотр примеров с самым маленьким context — гипотеза не подтвердилась
(можно узнать результат ниже), окончательное решение — в следующем разделе.

In [ ]:
summ = df_final[df_final['category'] == 'summarization'].copy()
summ['context_len'] = summ['context'].str.split().str.len()
summ['compression_ratio'] = summ['response_len'] / summ['context_len']

print(summ['compression_ratio'].describe())

bad_summaries = summ[summ['compression_ratio'] > 0.5]

# ratio > 0.5 means response retained more than half the original word count —
# suspicious for a task that's supposed to compress
print(f"\n Bad summaries (ratio > 0.5): {len(bad_summaries)} from {len(summ)}")
print(bad_summaries[['response_len', 'context_len', 'compression_ratio']].sort_values('compression_ratio', ascending=False).head(10))

# Tried splitting by context_len < 30 words as a proxy
# for "ratio is unreliable here" — but manual inspection below showed this
# threshold was arbitrary and didn't match the actual pattern in the data
# (see final approach in the next section: sentence count, not word count)
#print()
#print(bad_summaries['context_len'].describe())
#print(bad_summaries['context_len'].quantile([0.1, 0.25, 0.5, 0.75, 0.9]))

#print()
#tiny = bad_summaries[bad_summaries['context_len'] < 30]
#normal = bad_summaries[bad_summaries['context_len'] >= 30]

#print(f"Tiny context (unreliable ratio): {len(tiny)}")
#print(f"Normal context, but really poor compression:{len(normal)}")


# manual inspection of the smallest-context "bad" summaries —
# testing the hypothesis that short context makes the ratio unreliable
tiny_examples = bad_summaries.nsmallest(10, 'context_len')

for _, row in tiny_examples.iterrows():
    print(f"context_len={row['context_len']}, ratio={row['compression_ratio']:.2f}")
    print("CONTEXT:", row['context'])
    print("RESPONSE:", row['response'][:200])
    print("CATEGORY:",row['category'])



count    1186.000000
mean        0.857509
std         3.529291
min         0.005391
25%         0.172662
50%         0.346410
75%         0.885696
max       104.000000
Name: compression_ratio, dtype: float64

 Bad summaries (ratio > 0.5): 429 from 1186
       response_len  context_len  compression_ratio
12565           104            1         104.000000
634             317           11          28.818182
14186          1166           43          27.116279
1867           4274          195          21.917949
4786           1343           72          18.652778
13966           531           38          13.973684
1127            703           54          13.018519
11146           901           79          11.405063
1220            756           67          11.283582
6406            277           26          10.653846
context_len=1, ratio=104.00
CONTEXT: n/a
RESPONSE: The American Revolution was a five-year long war that started in 1775. The colonists of Great Britain were tired of being ta

## Финальное решение: структурный критерий вместо порога по длине

Ручной осмотр показал: проблема не в количестве слов в context, а в его
структуре. Примеры с самым маленьким context — это однопредложные
фактические утверждения ("X is a Y"), а response их разворачивал деталями.
Это структурно НЕ задача сжатия текста, а обратная задача (расширение
факта) — ошибочно попавшая в категорию summarization при разметке.

Итоговый критерий: context из ≤1 предложения — не настоящая summarization-
задача, удаляем полностью (а не просто исключаем из проверки ratio, как
предполагалось изначально). Отдельно удаляем "битые" context (заглушки
вида "n/a"). Ratio-фильтр (>0.5) применяется только к оставшимся,
подтверждённо многопредложным примерам.

In [ ]:
def count_sentences(text):
    # rough sentence count via punctuation splitting — good enough to
    # distinguish "one short factual statement" from "a real passage"
    if not text or not text.strip():
        return 0
    return len([s for s in text.replace('!', '.').replace('?', '.').split('.') if s.strip()])

summ['context_sentences'] = summ['context'].apply(count_sentences)

# placeholder values found during manual inspection (e.g. context="n/a")
# — data entry artifacts, not real content
is_placeholder = summ['context'].str.strip().str.lower().isin(['n/a', 'na', 'none', ''])
print(f"Broken context (n/a, etc.): {is_placeholder.sum()}")

# single-sentence context = not a genuine summarization task structurally
# (see markdown above for reasoning)
single_sentence = summ['context_sentences'] <= 1
print(f"Single sentence context (not a real summarization task): {single_sentence.sum()}")

summ_final = summ[~is_placeholder & ~single_sentence].copy()

# now that context is confirmed multi-sentence, high ratio reliably
# indicates a genuinely poor (uncompressed) summary
apply_ratio_filter = summ_final['compression_ratio'] > 0.5
summ_final = summ_final[~apply_ratio_filter]

print(f"Remaining summarizations: {len(summ_final)} out of {len(summ)}")


Broken context (n/a, etc.): 1
Single sentence context (not a real summarization task): 28
Remaining summarizations: 755 out of 1186


In [ ]:
# summarization already passed the structural filter above;
# open_qa/closed_qa haven't been touched since the initial exact-dedup step
other = df_final[df_final['category'] != 'summarization'].copy()

combined = pd.concat([other, summ_final], ignore_index=True)

print(len(combined))

6219


## Базовые эвристики качества

Применяем четыре простых, но содержательных фильтра ко всему объединённому
датафрейму разом:
- too_short — вероятный мусор/пустышка
- copies_instruction — модель/аннотатор не понял задание, просто повторил вопрос
- has_artefacts — артефакты форматирования (лишние пробелы/переносы от копипаста)
- response_eq_context — баг данных: ответ дословно скопирован из context
  вместо реального ответа на вопрос (найдено при ручном осмотре длинных
  response ранее)

In [ ]:
# likely junk/empty answers
too_short = combined['response_len'] < 3
print(f"Too short: {too_short.sum()}")

# response identical to the question itself — model/annotator didn't
# actually answer, just echoed the prompt
copies_instruction = combined.apply(
    lambda r: r['response'].strip().lower() == r['instruction'].strip().lower(), axis = 1
)
print(f"Response = Instruction: {copies_instruction.sum()}")

# formatting artifacts (e.g. excessive whitespace/line breaks from copy-paste)
has_artefacts = combined['response'].str.contains(r'\s{3,}', regex=True)
print(f"With artefacts: {has_artefacts.sum()}")

# data entry bug: response is a verbatim copy of context, not an actual answer
# (found earlier via manual inspection of the longest closed_qa/summarization responses)
response_eq_context = combined.apply(
    lambda r: bool(r['context']) and r['response'].strip() == r['context'].strip(), axis=1
)
print(f"Response = Context: {response_eq_context.sum()}")



df_quality = combined[~too_short & ~copies_instruction & ~has_artefacts & ~response_eq_context].copy()
print(f"Remaining after quality filtering: {len(df_quality)}")


Too short: 537
Response = Instruction: 0
With artefacts: 149
Response = Context: 6
Remaining after quality filtering: 5527


## Финальная выборка: 250 примеров

Берём 250 с запасом (не потому что "больше лучше",
а на случай непредвиденных потерь при дальнейшей обработке; в итоге запас
не понадобился, чистых данных осталось с большим избытком).

Пропорции по категориям: open_qa — 100 (ключевая категория для демонстрации
стиля "эксперт", наиболее показательна для style transfer), summarization — 80
(средняя по важности), closed_qa — 70 (самая узкая по исходной доступности
данных, взята чуть скромнее). random_state=42 — для воспроизводимости отбора.

In [ ]:
def sample_category(df, cat, n, seed=42):
    sub = df[df['category'] == cat]
    return sub.sample(n=n, random_state=seed)

open_qa_final = sample_category(df_quality, 'open_qa', 100)
closed_qa_final = sample_category(df_quality, 'closed_qa', 70)
summarization_final = sample_category(df_quality, 'summarization', 80)

final_dataset = pd.concat([open_qa_final, closed_qa_final, summarization_final], ignore_index=True)
print(final_dataset['category'].value_counts())
print(f"Total: {len(final_dataset)}")

category
open_qa          100
summarization     80
closed_qa         70
Name: count, dtype: int64
Total: 250


# Style Transfer: синтез датасета в целевом стиле

Инструкции берём как есть из очищенного датасета — они уже разнообразны
и качественны. Переписываем только response через LLM API в стиль
"лаконичный эксперт": сохраняем факты, убираем многословность и оговорки.

Модель: Llama 3.3 70B через Groq API (бесплатно, без карты, щедрые лимиты —
30 запросов/мин, 1000/день). Промпт на английском (данные на английском —
снижает риск переключения языка/несогласованности стиля).

Явный запрет на списки/markdown в правиле 5 — добавлен после того, как
на тестовом прогоне модель самостоятельно добавила буллет-список в одном
из 5 примеров, хотя в оригинале его не было (нарушение консистентности
формата, важной для обучения на маленьком датасете).

In [ ]:
!pip install groq

from groq import Groq
from google.colab import userdata
import time

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def rewrite_response(instruction, context, response,  max_retries=3):
    context_block = f"\nContext: {context}" if context else ""

    prompt = f"""You rewrite an expert's answer in a concise style. Rules:
1. Preserve ALL factual details from the original — do not omit anything meaningful.
2. Remove filler phrases, hedging, and repetition.
3. Write directly and confidently, without unnecessary padding.
4. Do not add anything that wasn't in the original.
5. Always respond in plain prose — do not use bullet points, numbered lists, or any markdown formatting.

Question: {instruction}{context_block}
Original answer: {response}

Rewrite the answer (output ONLY the rewritten text, no explanations):"""

    # retry with backoff — free tier rate limits are occasionally hit
    # even with sleep() between calls, so this prevents a single 429
    # from killing the entire batch run
    for attempt in range(max_retries):
        try:
            completion = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,
            )
            return completion.choices[0].message.content
        except Exception as e:
            if '429' in str(e) and attempt < max_retries - 1:
                wait = 15 * (attempt + 1)
                print(f"Rate limit, wait {wait}s...")
                time.sleep(wait)
            else:
                raise

# quick manual test on 5 examples before running the full 250 —
# cheaper to catch a bad prompt now than after burning the full API quota
test_sample = final_dataset.head(5)
for _, row in test_sample.iterrows():
    new_response = rewrite_response(row['instruction'], row['context'], row['response'])

    before_len = len(row['response'].split())
    after_len = len(new_response.split())


    print("INSTRUCTION:", row['instruction'][:100])
    print(f"BEFORE ({before_len} words):", row['response'])
    print(f"AFTER: ({after_len} words):", new_response)
    print(
    f"Length change: {after_len - before_len:+d} words ({after_len / before_len * 100:.0f}% of original)")
    print("---")

<font size="2">
INSTRUCTION: When did the first World war start?
<br>
BEFORE (3 words): July 28, 1914
<br>
AFTER: (3 words): July 28, 1914
<br>
Length change: +0 words (100% of original)
<br>
---
</font><br>

<font size="2">
INSTRUCTION: Give me some facts about Ethiopia?
<br>
BEFORE (138 words): Ethiopia is a federal parliamentary republic with a population of 113,656,596 people, as of 2021 (World factbook). The country is led by Prime Minister Abiy Ahmed, who came to power in 2018. Voting is allowed for citizens over 18 years of age, but it is not mandatory. Despite its leadership in government, Ethiopia faces significant issues with corruption. According to a corruption scale, the country ranks 129th out of 180 countries, with bribery being a common occurrence in government affairs. Agriculture is a major part of Ethiopia's economy, with 38.5 million hectares of land being cultivated in 2020(World factbook). The major crops grown in the country include coffee, pulses, oilseeds, cereals, potatoes, sugarcane, and vegetables. Approximately 80% of Ethiopia's population resides in rural areas due to the need for farming, while the remaining 20% lives in urban areas.
<br>AFTER: (83 words): Ethiopia is a federal parliamentary republic with a 2021 population of 113,656,596. Prime Minister Abiy Ahmed has led the government since 2018. Citizens over 18 may vote, but voting is not compulsory. The country ranks 129th of 180 on a corruption index, with bribery common in government affairs. Agriculture dominates the economy; 38.5 million hectares were cultivated in 2020, producing coffee, pulses, oilseeds, cereals, potatoes, sugarcane and vegetables. Roughly 80 % of Ethiopians live in rural areas and 20 % in urban areas.
<br>Length change: -55 words (60% of original)
<br>
------</font><br>

<font size="2">
INSTRUCTION: Name three national parks in California.
...
Complex issues arise in spatial analysis, many of which are neither clearly defined nor completely resolved, but form the basis for current research. The most fundamental of these is the problem of defining the spatial location of the entities being studied. Classification of the techniques of spatial analysis is difficult because of the large number of different fields of research involved, the different fundamental approaches which can be chosen, and the many forms the data can take.
<br>AFTER: (87 words): Spatial analysis comprises formal techniques that examine entities by their topological, geometric, or geographic properties, often employing spatial statistics. It is used across disciplines—from astronomy’s study of galaxy placement to chip fabrication’s “place and route” algorithms for wiring, and in genomics such as transcriptomics. In a narrower sense, it refers to geospatial analysis of human‑scale structures, especially geographic data. The field faces unresolved issues, foremost the definition of entities’ spatial locations, and classification is challenging because of the diversity of research fields, methodological approaches, and data forms.
<br>Length change: -99 words (47% of original)
<br>
---<br>
</font>

## Полный прогон на 250 примерах

Пишем результат построчно (не накапливаем всё в памяти и не сохраняем
одним файлом в конце) — если прогон прервётся на середине (сбой сети,
превышение квоты и т.д.), уже обработанные примеры не потеряются.

try/except вокруг каждого примера — единичная ошибка не должна обрушить
весь батч из 250 запросов; проблемный пример просто пропускается,
остальные продолжают обрабатываться.

In [ ]:
import json

results = []
output_path = 'style_transfer_progress.jsonl'

with open(output_path, 'w') as f:
    for i, (_, row) in enumerate(final_dataset.iterrows()):
        try:
            new_response = rewrite_response(row['instruction'], row['context'], row['response'])
            record = {
                'instruction': row['instruction'],
                'context': row['context'],
                'original_response': row['response'],
                'response': new_response.strip(),
                'category': row['category']
            }
            results.append(record)

             # write immediately + flush — survives a crash mid-run without losing progress
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            f.flush()

            if (i + 1) % 25 == 0:
                print(f"Processed {i+1}/{len(final_dataset)}")

                # ~30 requests/min free tier limit — this pace stays safely under it
                time.sleep(2.5)
        except Exception as e:
            # skip this example, don't kill the whole 250-example run over one failure
            print(f"Error in the example {i} ({row['instruction'][:50]}): {e}")
            time.sleep(10)

print(f"Ready: {len(results)} of {len(final_dataset)}")

Processed 25/250
Processed 50/250
Processed 75/250
Rate limit, жду 15с...
Processed 100/250
Rate limit, жду 15с...
Processed 125/250
Rate limit, жду 15с...
Processed 150/250
Processed 175/250
Processed 200/250
Processed 225/250
Rate limit, жду 15с...
Processed 250/250
Ready: 250 of 250


## Валидация результата style transfer

После полного прогона на 250 примерах — проверяем целостность (нет ли
дублей, которые могли возникнуть из-за случайного повторного запуска
ячейки) и категориальный баланс, а затем визуально просматриваем случайную
выборку по всему датасету (не только первые несколько, которые уже видели
на этапе теста промпта) — чтобы убедиться в стабильном качестве, а не
только на "удачных" примерах в начале.

In [ ]:
import json
import random

with open('style_transfer_progress.jsonl', 'r') as f:
    records = [json.loads(line) for line in f]

print(f"Total records: {len(records)}")


# sanity check: duplicates would indicate the write loop ran twice
# (e.g. accidental re-run of the same cell) on the same output file
instructions = [r['instruction'] for r in records]
print(f"Instruction duplicates: {len(instructions) - len(set(instructions))}")
print(f"Categories: {pd.Series([r['category'] for r in records]).value_counts().to_dict()}")

# random (not first-N) sample — avoids only checking the "easy" early examples
random.seed(42)
sample = random.sample(records, 8)

for r in sample:
    before_len = len(r['original_response'].split())
    after_len = len(r['response'].split())
    print(f"[{r['category']}] {r['instruction'][:80]}")
    print(f"  Before ({before_len}): {r['original_response'][:150]}")
    print(f"  After ({after_len}): {r['response']}")
    print()

Total records: 250
Instruction duplicates: 0
Categories: {'open_qa': 100, 'summarization': 80, 'closed_qa': 70}
[closed_qa] What is an onigiri made of?
  Before (55): Onigiri is a Japanese food made from white rice formed into triangular or cylindrical shapes and often wrapped in nori (seaweed). Traditionally, an on
  After (51): Onigiri is a Japanese rice ball made from white rice shaped into triangles or cylinders, often wrapped in nori. Traditional fillings include pickled ume (umeboshi), salted salmon, katsuobushi, kombu, tarako, mentaiko, takanazuke (pickled takana, Japanese giant red mustard greens) or any other salty or sour ingredient used as a natural preservative.

[open_qa] What does zan zendegi azadi mean?
  Before (10): Zan zendegi azadi translates from Farsi to woman, life, freedom.
  After (9): Zan zendegi azadi means woman, life, freedom in Farsi.

[open_qa] Name different corporate messaging applications companies use.
  Before (11): Corporate messaging applications co

## Поиск возможных "додумываний" модели

Правило 4 промпта запрещает добавлять факты, которых не было в оригинале —
но полную ручную проверку всех 250 примеров провести нереально. Ищем
прокси-сигнал: короткие оригинальные ответы (мало "материала" для
интерпретации), которые сильно выросли в объёме после переписывания —
это подозрительно, поскольку для style transfer в сторону лаконичности
рост объёма нетипичен и обычно означает, что модель что-то домыслила
сверх исходного текста.

In [ ]:
# heuristic: a short original answer growing 50%+ after a "make it concise"
# rewrite is suspicious — it likely means the model added information
# not present in the source, rather than reformatting existing content
suspicious = []
for r in records:
    before_words = len(r['original_response'].split())
    after_words = len(r['response'].split())
    if before_words <= 10 and after_words > before_words * 1.5:
        suspicious.append(r)

print(f"Suspicious cases: {len(suspicious)}")
for r in suspicious:
    print(f"[{r['category']}] {r['instruction'][:80]}")
    print(f"  Before: {r['original_response']}")
    print(f"  After: {r['response']}")
    print()

Suspicious cases (short original, heavily expanded): 6 <br>
[open_qa] Where was 52nd International Film Festival of India held ?<br>
BEFORE: Goa (Nov 20 to Nov 28)<br>
AFTER: It was held in Goa from November 20 to November 28.<br>

[open_qa] What is Orchesis - either professional or amateur<br>
BEFORE: Art of Dancing<br>
AFTER: Orchesis is the art of dancing.<br>

[open_qa] Greta Thunberg is a Swedish Environmental activist has donated the  Prize  money<br>
BEFORE: Environment and climate change<br>
AFTER: She donated the prize money to support environmental and climate-change initiatives.<br>

[closed_qa] Which English football club has won the most FA Cup trophies?<br>
BEFORE: Arsenal - 14<br>
AFTER: Arsenal hold the record with 14 FA Cup wins, one of only six clubs to win the competition in consecutive seasons (2002-03 and 2014-15). They have completed three league-and-FA Cup doubles (1971, 1998, 2002), a feat matched only by Manchester United (1994, 1996, 1999). Arsenal were the first English side to achieve an FA Cup-League Cup double in 1993 and the first London club to reach a UEFA Champions League final, losing 2-1 to Barcelona in 2006.<br>

[closed_qa] What are the official languages of the United Nations?<br>
BEFORE: Arabic, Mandarin Chinese, English, French, Russian, Spanish<br>
AFTER: The United Nations’ official languages are Arabic, Mandarin Chinese, English, French, Russian, and Spanish.<br>

[summarization] provide the list of the List of American foods that is related to breeds<br>
BEFORE: Banana bread<br>
Cornbread
Cuban bread
Frybread
Texas toast
Cheese
AFTER: The related American foods are banana bread, cornbread, Cuban bread, frybread, Texas toast, and cheese.<br>

## Точечное исправление найденной ошибки

Единственный подтверждённый случай добавления фактов, которых не было
в оригинале (проверка выше) — ответ про FA Cup, где модель добавила
несуществующие в исходном тексте детали (про "doubles", финал Лиги
чемпионов). Оригинал содержал только два факта: клуб и число трофеев —
исправляем вручную, сохраняя эти два факта в кратком виде, без остального
неподтверждённого текста.

In [ ]:
for r in records:
    if r['instruction'] == 'Which English football club has won the most FA Cup trophies?':
        r['response'] = "Arsenal have won the most FA Cup trophies, with 14 titles."
        print("Fixed:", r['response'])


with open('final_style_transfer.jsonl', 'w') as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

Fixed: Arsenal have won the most FA Cup trophies, with 14 titles.


## Сохранение результата в Google Drive

Финальный датасет (250 примеров, style transfer завершён) сохраняем
в Drive — данные переживут переключение runtime на GPU для следующего
шага (fine-tuning), в отличие от файлов в /content/, которые исчезают
при смене или истечении сессии Colab.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!cp final_style_transfer.jsonl /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
